### Setup Notebook

In [ ]:
# # Cell 1: Uninstall all related packages
# %pip uninstall langchain langchain-core langchain-community langchain-openai langchain-neo4j -y


In [ ]:
# # # Cell 2: Restart the Kernel
# # # # This step is critical to clear the old packages from memory
# import IPython
# IPython.Application.instance().kernel.do_shutdown(True)

In [1]:
# Cell 3: Reinstall modern, compatible packages
# %pip install langchain-core langchain-community langchain-openai openai neo4j langgraph

In [2]:
# import json
# from tqdm import tqdm
from dotenv import load_dotenv
load_dotenv()

import os

from operator import itemgetter 

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langchain_community.graphs import Neo4jGraph
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser




from typing_extensions import List, TypedDict
from typing import Dict, Any


import psycopg2
from neo4j import GraphDatabase
from decimal import Decimal
from shapely import wkb
from pyproj import CRS, Transformer


import pandas as pd


C:\Users\Syed Haque\AppData\Roaming\Python\Python314\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [3]:

load_dotenv()

keys = ["NEO4J_USER_GROUP", "NEO4J_PASSWORD_GROUP","NEO4J_USER_LOCAL", "NEO4J_PASSWORD_LOCAL"]
for key in keys:
    if key not in os.environ:
        raise Exception(f"Key '{key}' not found not in .env")
    
print("Credentials from .env file successfully loaded")

NEO4J_LOCAL_DATABASE = "dse203test"


class Config:
    def __init__(self, mode="LOCAL", DATABASE='neo4j'):
        mode = mode.upper()

        if mode == "LOCAL":
            self.URI = os.getenv("NEO4J_URI_LOCAL")
            self.USER = os.getenv("NEO4J_USER_LOCAL")
            self.PASSWORD = os.getenv("NEO4J_PASSWORD_LOCAL")
        elif mode == "GROUP":
            self.URI = os.getenv("NEO4J_GROUP_URI")
            self.USER = os.getenv("NEO4J_GROUP_USER")
            self.PASSWORD = os.getenv("NEO4J_GROUP_PASSWORD")
        else:
            raise ValueError("Mode must be 'LOCAL' or 'GROUP'.")
        self.DATABASE = DATABASE


config = Config(mode="LOCAL",DATABASE = NEO4J_LOCAL_DATABASE)
driver = GraphDatabase.driver(config.URI, auth=(config.USER, config.PASSWORD))


Credentials from .env file successfully loaded


In [4]:
graph = Neo4jGraph(url=config.URI,    username=config.USER,password=config.PASSWORD,database=config.DATABASE )
# graph = Neo4jGraph(url="bolt://67.58.49.87:7687",    username='neo4j',password="h2u9l4px" )
graph.query("MATCH (n) RETURN n LIMIT 1;")




C:\Users\Syed Haque\AppData\Local\Temp\ipykernel_18264\3917807703.py:1: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(url=config.URI,    username=config.USER,password=config.PASSWORD,database=config.DATABASE )


[{'n': {'businesses_food_beverage_stores_naics': 1.0,
   'sales_wholesale_trade_naics': 0.0,
   'income_200000_plus': 3,
   'businesses_insurance_sic': 0.0,
   'businesses_finance_insurance_realestate_sic': 4.0,
   'geom': '0106000020E6100000010000000103000000010000004A000000482F0D3A21445DC0349513F88A4C40409F88A93121445DC09D4E445B954C4040C0E4772D21445DC0B39576D4984C40407F6C462921445DC07279FFA4A14C40401091462921445DC0B74D2838BC4C40408798452921445DC058201411C54C4040F669152521445DC01E3B8BE6C84C40403DAC152521445DC0E798B977D44C4040C7DC142521445DC05A9E9355D84C4040A9D7E32021445DC0378B686AD94C40402E0F4E1421445DC0B7B46DCADC4C40404AB44E1421445DC0C682A5E7DD4C4040A7244E1421445DC0B28D8353E64C404073927F1821445DC088EA21DBF94C40407CAF7F1821445DC045734DB0FF4C404083F4B11C21445DC06B648E24084D4040B95D32772D445DC0B2747B410B4D40408BDFA68636445DC0E7D212950D4D40408B2102F73C445DC0D02C39730F4D4040B9EF4DBF44445DC07D14B483114D4040C1647E9752445DC09A8DE565144D4040E6A294FE5E445DC0B0AAFC04174D404077E522FE61445DC01F4E

In [6]:
get_schema_runnable = graph.get_schema

In [7]:
# Load the CSV file
node_attributes_df = pd.read_csv('../data/llm/business_opportunity_entity_attributes.csv')
node_edges_df = pd.read_csv('../data/llm/business_opportunity_entity_relationships.csv')


def csv_nodes_to_schema(node_attributes_df):
    node_blocks = []

    for entity, group in node_attributes_df.groupby("Entity"):
        lines = []
        
        for _, row in group.iterrows():
            attr = row["Attribute"]
            t = row["Type"]
            desc = row["Description"]

            lines.append(f"  - {attr} ({t}): {desc}")

        block = f"{entity}:\n" + "\n".join(lines)
        node_blocks.append(block)

    return "\n\n".join(node_blocks)


def csv_relationships_to_schema(node_edges_df):
    rel_blocks = []

    for _, row in node_edges_df.iterrows():
        rel_blocks.append(
            f"{row['Entity Type 1']} --[{row['Relationship']}]→ {row['Entity Type 2']}"
        )

    return "\n".join(rel_blocks)


In [8]:
neo4j_schema = graph.get_schema 

merged_schema = f"""
=== Neo4j Schema ===
{neo4j_schema}

=== CSV Entity Descriptions ===
{csv_nodes_to_schema(node_attributes_df)}

=== CSV Relationship Patterns ===
{csv_relationships_to_schema(node_edges_df)}
"""


In [9]:
merged_schema

'\n=== Neo4j Schema ===\nNode properties:\nBlockGroup {id: FLOAT, businesses_food_beverage_stores_naics: FLOAT, sales_wholesale_trade_naics: FLOAT, income_200000_plus: INTEGER, businesses_insurance_sic: FLOAT, businesses_finance_insurance_realestate_sic: FLOAT, geom: STRING, population_70_79: INTEGER, businesses_other_services_sic: FLOAT, male_population_70_79: INTEGER, sales_legal_services_sic: FLOAT, businesses_health_services_sic: FLOAT, female_population_0_9: INTEGER, sales_agriculture_naics: FLOAT, businesses_hotels_sic: FLOAT, population_80_plus: INTEGER, female_population_50_59: INTEGER, income_15000_24999: INTEGER, sales_finance_insurance_realestate_sic: FLOAT, businesses_accommodation_naics: FLOAT, population_20_29: INTEGER, male_population_50_59: INTEGER, income_75000_99999: INTEGER, sales_hotels_sic: FLOAT, female_population_70_79: INTEGER, male_population_20_29: INTEGER, businesses_wholesale_trade_sic: FLOAT, population_50_59: INTEGER, sales_total_naics: FLOAT, ctblockgroup

### Setup LLM Chat

In [10]:
# Initialize the LLM
model_ver = "gpt-4o"

cypher_model = ChatOpenAI(model="gpt-4o-mini", temperature=0) 

main_qa_model = ChatOpenAI(model="gpt-4o", temperature=0)

#### Generic GraphRAG

In [11]:
# Define state for application
class State(TypedDict):
    question: str
    context: List[dict]
    answer: str


# Retrieve context 
# def retrieve(state: State):
#     context = graph.query("CALL db.schema.visualization()")
#     return {"context": context}



def retrieve(state: State):
    schema_map = graph.query("CALL apoc.meta.schema()")
    
    context = schema_map 
    return {"context": context}

# Create a prompt
template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

{context}

Question: {question}

Answer:"""




prompt = ChatPromptTemplate.from_template(template)

# Generate the answer based on the question and context
def generate(state: State):
    messages = prompt.invoke({"question": state["question"], "context": state["context"]})
    response = main_qa_model.invoke(messages)
    return {"answer": response.content}




# Define application steps
workflow = StateGraph(State).add_sequence([retrieve, generate])
workflow.add_edge(START, "retrieve")
app = workflow.compile()




In [12]:
# entity_type = 'BlockGroup'
# question = "How is the graph structured?"



question = "What questions can you answer"
# question = f"What attributes are available on a {entity_type} node"
# question = "Are any cities contained in cities"

# question = 'What is the context'

response = app.invoke({"question": question})
print("Answer:", response["answer"])

Answer: I can answer questions related to the structure and relationships within the provided data context. Here are some examples of questions I can answer:

1. What types of nodes and relationships are present in the data?
2. How many nodes exist for each type (e.g., Community, Brand, BlockGroup)?
3. What properties are associated with each node type?
4. What relationships exist between different node types, and what are their properties?
5. How many relationships exist for each type (e.g., ADJACENT_TO, NEARBY)?
6. What are the unique or indexed properties for specific node types?
7. How are nodes like BusinessLocation related to other nodes like Brand or ZoneLocation?

If you have specific questions about the data structure or relationships, feel free to ask!


####  Text 2 Cypher

In [13]:
# Test 1: Check if 'query' is callable
print(f"Type of graph.query: {type(graph.query)}")

# Test 2: Run a simple query
try:
    test_result = graph.query("MATCH (n) RETURN count(n) AS count")
    print("Graph query test passed.")
    print(test_result)
except AttributeError as e:
    print(f"Error during query test: {e}")

Type of graph.query: <class 'method'>
Graph query test passed.
[{'count': 78025}]


In [14]:


# from the online course
# cypher_qa = GraphCypherQAChain.from_llm(
#     graph=graph, 
#     llm=model, 
#     cypher_llm=cypher_model,
#     allow_dangerous_requests=True,
#     verbose=True,
# )


# # Create the Cypher QA chain
# cypher_qa = GraphCypherQAChain.from_llm(
#     graph=graph, 
#     llm=model, 
#     allow_dangerous_requests=True,
#     verbose=True, 
# )



In [ ]:
neo4j_schema = graph.get_schema
print("Schema successfully retrieved.")
print(neo4j_schema)

Schema successfully retrieved.
Node properties:
BlockGroup {id: FLOAT, businesses_food_beverage_stores_naics: FLOAT, sales_wholesale_trade_naics: FLOAT, income_200000_plus: INTEGER, businesses_insurance_sic: FLOAT, businesses_finance_insurance_realestate_sic: FLOAT, geom: STRING, population_70_79: INTEGER, businesses_other_services_sic: FLOAT, male_population_70_79: INTEGER, sales_legal_services_sic: FLOAT, businesses_health_services_sic: FLOAT, female_population_0_9: INTEGER, sales_agriculture_naics: FLOAT, businesses_hotels_sic: FLOAT, population_80_plus: INTEGER, female_population_50_59: INTEGER, income_15000_24999: INTEGER, sales_finance_insurance_realestate_sic: FLOAT, businesses_accommodation_naics: FLOAT, population_20_29: INTEGER, male_population_50_59: INTEGER, income_75000_99999: INTEGER, sales_hotels_sic: FLOAT, female_population_70_79: INTEGER, male_population_20_29: INTEGER, businesses_wholesale_trade_sic: FLOAT, population_50_59: INTEGER, sales_total_naics: FLOAT, ctblock

In [ ]:

# --- 2. Prompt Templates ---



CYPHER_PROMPT = ChatPromptTemplate.from_messages(
    [
        ("system", 
         "You are an expert Neo4j developer. Use the detailed schema: {schema}. "
         "**CRITICAL RULE:** The `location` attribute is already a Neo4j POINT object. **NEVER** use the `point()` function on `n.location`. "
         "**GEOSPATIAL RULE:** To find neighbors, calculate distance using `point.distance(p1.location, p2.location)` in meters (0.1 degrees is 11132m). "
         "**CYPHER SCOPE RULE:** Always use `WITH` to carry forward necessary node variables (like the original location node, 'bl') into subsequent clauses. "
         "Generate a single, correct, executable Cypher query. **ONLY output the raw Cypher statement.**"
         "** LIMIT output to 10 unless specified otherwise"
         "**CYPHER OUTPUT RULE: You MUST return intermediate match results in the final RETURN statement.**"),
        ("human", "Question: {question}"),
        ("system", "BEGIN CYPHER QUERY. DO NOT ADD ANY OTHER TEXT."),
    ]
)

ANSWER_PROMPT = ChatPromptTemplate.from_messages(
    [
        ("system", 
         "You are a helpful assistant. Use the Query Result: {query_result},Original Question: {question} and Generated Query:{generated_cypher}"
         "to provide a concise answer."
         "**If you are unsure, look at the Generated Query {generated_cypher}, the location asked about might be an intermediate step.**"
         ),
        ("human", "Original Question: {question}"),
    ]
)

# --- 3. Helper Functions for LCEL ---

def prepare_input_with_schema(input_dict: Dict[str, Any]) -> Dict[str, Any]:
    """Injects the static schema string into the incoming question dictionary."""
    return {
        "question": input_dict["question"],
        "schema": merged_schema # Injects the pre-fetched static string
    }

def execute_cypher_query(input_dict: Dict[str, Any]) -> str:
    """Executes the generated Cypher query using the Neo4jGraph object."""
    return graph.query(input_dict['cypher'])

# --- 4. The Final LCEL Chain ---

cypher_qa_chain = (
    # 1. Start by injecting the static schema string
    RunnableLambda(prepare_input_with_schema)
    | {
        # 2. Parallel Step: Generate Cypher, and PASS THROUGH question/schema
        "cypher": CYPHER_PROMPT | cypher_model | StrOutputParser(),
        "question": itemgetter("question"),
        "schema": itemgetter("schema"), 
    }
    # 3. Execution Step: Pass generated Cypher through and execute query
    | RunnablePassthrough.assign(
        query_result=RunnableLambda(execute_cypher_query),
        question=itemgetter("question"),
        generated_cypher=itemgetter("cypher"), 
    )
    # 4. Final Output Formatting: Define the exact keys to return
    | {
        "final_answer": ANSWER_PROMPT | main_qa_model | StrOutputParser(),
        "generated_cypher": itemgetter("generated_cypher"), 
        
        # *** ADDED: Return the raw Cypher query output ***
        "query_output": itemgetter("query_result") 
    }
)

# Example Invocation and Output Access:
# question = "Find a starbucks with a high rating..."
# response = cypher_qa_chain.invoke({"question": question})

# print(f"1. Final Answer: {response['final_answer']}")
# print(f"2. Cypher Query: {response['generated_cypher']}")
# print(f"3. Query Output: {response['query_output']}")


In [85]:

question = "Find a starbucks with a high rating, look for its location, create a boundaing box of +/- 0.1 degrees and return 5 business locations in there"
response = cypher_qa_chain.invoke({"question": question})

print(f"1. Final Answer: {response['final_answer']}")
print(f"2. Cypher Query: {response['generated_cypher']}")
print(f"3. Query Output: {response['query_output']}")


1. Final Answer: One Starbucks with a high rating is located at McIntire Dr, San Diego, CA 92134, with an average rating of 5.0. The coordinates for this location are approximately 32.7275765 latitude and -117.1453109 longitude.

Using these coordinates, a bounding box of +/- 0.1 degrees would cover the area between:

- Latitude: 32.6275765 to 32.8275765
- Longitude: -117.2453109 to -117.0453109

Within this bounding box, here are 5 business locations:

1. **Starbucks**
   - Address: McIntire Dr, San Diego, CA 92134
   - Categories: Coffee shop
   - Rating: 5.0

2. **Fleet Science Center**
   - Address: 1875 El Prado, San Diego, CA 92101
   - Categories: Science museum, IMAX theater, Tourist attraction
   - Rating: 4.5

3. **The Silver Fox Lounge**
   - Address: 1833 Garnet Ave, San Diego, CA 92109
   - Categories: Bar
   - Rating: 4.4

4. **False Idol**
   - Address: 675 W Beech St, San Diego, CA 92101
   - Categories: Bar
   - Rating: 4.7

5. **Plaza Chula Vista**
   - Address: 1210 

In [2]:

question = "What zone locations contain Coffee shops (subsector name)"
response = cypher_qa_chain.invoke({"question": question})

print(f"1. Final Answer: {response['final_answer']}")
print(f"2. Cypher Query: {response['generated_cypher']}")
print(f"3. Query Output: {response['query_output']}")

NameError: name 'cypher_qa_chain' is not defined

#### Ask Questions

In [ ]:
question = "What zones are coffee shop (subsector name might have this initial capitalized) located in"
response = cypher_qa_chain.invoke({"question": question})

print(f"1. Final Answer: {response['final_answer']}")
print(f"2. Cypher Query: {response['generated_cypher']}")
print(f"3. Query Output: {response['query_output']}")


In [31]:
# Invoke the chain
question = "Find a starbucks with a high rating, look for its location, create a boundaing box of +/- 0.1 degrees and return 5 business locations in there"
response = cypher_qa_chain.invoke({"question": question}) 

print(response["final_answer"])

The provided data does not contain any Starbucks locations, so I am unable to find a Starbucks with a high rating or its location. Therefore, I cannot create a bounding box or return business locations within that area. The information is unavailable based on the current data.


In [32]:
# Invoke the chain
question = "What kind of business locations have high ratings in heavily tot populated blockgroups where starbucks" \
" have high ratings? " \
"Find 5 of those businesses that do well but in a blockgroup with few starbucks"
response = cypher_qa_chain.invoke({"question": question})
print(response["final_answer"])

The information regarding business locations with high ratings in heavily populated blockgroups where Starbucks also have high ratings, and identifying 5 such businesses that do well in blockgroups with few Starbucks, is unavailable.


In [33]:
print(graph.schema)

Node properties:
BlockGroup {id: FLOAT, businesses_food_beverage_stores_naics: FLOAT, sales_wholesale_trade_naics: FLOAT, income_200000_plus: INTEGER, businesses_insurance_sic: FLOAT, businesses_finance_insurance_realestate_sic: FLOAT, geom: STRING, population_70_79: INTEGER, businesses_other_services_sic: FLOAT, male_population_70_79: INTEGER, sales_legal_services_sic: FLOAT, businesses_health_services_sic: FLOAT, female_population_0_9: INTEGER, sales_agriculture_naics: FLOAT, businesses_hotels_sic: FLOAT, population_80_plus: INTEGER, female_population_50_59: INTEGER, income_15000_24999: INTEGER, sales_finance_insurance_realestate_sic: FLOAT, businesses_accommodation_naics: FLOAT, population_20_29: INTEGER, male_population_50_59: INTEGER, income_75000_99999: INTEGER, sales_hotels_sic: FLOAT, female_population_70_79: INTEGER, male_population_20_29: INTEGER, businesses_wholesale_trade_sic: FLOAT, population_50_59: INTEGER, sales_total_naics: FLOAT, ctblockgroup: INTEGER, sales_education